### Hybrid Frequency-Spatial Transformer with Physics-Guided Fusion


## Step 1 — Check GPU & Install Dependencies

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found! Go to Settings → Accelerator → GPU T4 x2")

!pip install timm einops scikit-image -q
print("Dependencies ready")

## Step 2 — Explore Dataset Paths
To find the EXACT folder structure of datasets.

In [ ]:
import os

# /kaggle/input 
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level >= 2:  # avoid spam
        sub = '  ' * (level + 1)
        shown = files[:5]
        for f in shown:
            print(f'{sub}{f}')
        if len(files) > 5:
            print(f'{sub}... ({len(files)} files total)')

## Step 3 — Set Dataset Paths
Based on Step 2 output, set the correct paths below.

In [ ]:
import os

RAW_ROOT = '/kaggle/input/datasets/larjeck/uieb-dataset-raw/raw-890/'
REF_ROOT = '/kaggle/input/datasets/larjeck/uieb-dataset-reference/reference-890/'

input_base = '/kaggle/input'
for d in os.listdir(input_base):
    full = os.path.join(input_base, d)
    if 'raw' in d.lower() and RAW_ROOT is None:
        RAW_ROOT = full
    if ('reference' in d.lower() or 'ref' in d.lower()) and REF_ROOT is None:
        REF_ROOT = full

print("RAW images root :", RAW_ROOT)
print("REF images root :", REF_ROOT)

if RAW_ROOT is None or REF_ROOT is None:
    print("\n  Could not auto-detect paths. Set RAW_ROOT and REF_ROOT manually above.")
    print("    Available folders:", os.listdir(input_base))
else:
   
    print("\nFolders inside RAW_ROOT:", os.listdir(RAW_ROOT))
    print("Folders inside REF_ROOT:", os.listdir(REF_ROOT))

## Step 4 — Build Working Directory & Generate MLLE Priors

In [ ]:
import os, shutil, random
from pathlib import Path
from PIL import Image, ImageEnhance
import numpy as np

WORK = '/kaggle/working/uieb-dataset'
for split in ['train', 'val', 'test']:
    for sub in ['INPUT', 'GT', 'PRIOR']:
        Path(f'{WORK}/{split}/{sub}').mkdir(parents=True, exist_ok=True)

def collect_images(folder):
    """Recursively find all images under folder."""
    imgs = []
    if folder is None or not os.path.exists(folder):
        return imgs
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                imgs.append(os.path.join(root, f))
    return sorted(imgs)

raw_imgs = collect_images(RAW_ROOT)
ref_imgs = collect_images(REF_ROOT)
print(f"Found {len(raw_imgs)} raw images and {len(ref_imgs)} reference images")

raw_by_name = {os.path.basename(p): p for p in raw_imgs}
ref_by_name = {os.path.basename(p): p for p in ref_imgs}
common_names = sorted(set(raw_by_name) & set(ref_by_name))
print(f"Matched pairs: {len(common_names)}")

if len(common_names) == 0:
    print("\n  No filename matches! Trying to pair by index order instead.")
    n = min(len(raw_imgs), len(ref_imgs))
    pairs = [(raw_imgs[i], ref_imgs[i]) for i in range(n)]
else:
    pairs = [(raw_by_name[n], ref_by_name[n]) for n in common_names]

random.seed(42)
random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
splits  = {
    'train': pairs[:n_train],
    'val'  : pairs[n_train:n_train+n_val],
    'test' : pairs[n_train+n_val:]
}
for s, p in splits.items():
    print(f"  {s}: {len(p)} pairs")

for split, pair_list in splits.items():
    for raw_p, ref_p in pair_list:
        fname = os.path.basename(raw_p)
        shutil.copy2(raw_p, f'{WORK}/{split}/INPUT/{fname}')
        shutil.copy2(ref_p, f'{WORK}/{split}/GT/{fname}')

print("\n Images copied to working directory")

In [ ]:
from tqdm.notebook import tqdm

def generate_mlle_prior(img_pil):
    """MLLE approximation — gray-world WB + contrast + gamma blend."""
    img = np.array(img_pil).astype(np.float32)
    mr, mg, mb = img[:,:,0].mean(), img[:,:,1].mean(), img[:,:,2].mean()
    mall = (mr + mg + mb) / 3.0
    img[:,:,0] = np.clip(img[:,:,0] * mall / (mr + 1e-6), 0, 255)
    img[:,:,1] = np.clip(img[:,:,1] * mall / (mg + 1e-6), 0, 255)
    img[:,:,2] = np.clip(img[:,:,2] * mall / (mb + 1e-6), 0, 255)
    img_wb = Image.fromarray(img.astype(np.uint8))
    img_contrast = ImageEnhance.Contrast(img_wb).enhance(1.5)
    gamma = 0.8
    img_gamma = np.power(np.array(img_contrast).astype(np.float32)/255.0, gamma)
    result = (0.4*np.array(img_wb).astype(np.float32)/255.0
            + 0.3*np.array(img_contrast).astype(np.float32)/255.0
            + 0.3*img_gamma)
    return Image.fromarray((np.clip(result, 0, 1)*255).astype(np.uint8))

for split in ['train', 'val', 'test']:
    input_dir  = Path(f'{WORK}/{split}/INPUT')
    output_dir = Path(f'{WORK}/{split}/PRIOR')
    files = list(input_dir.glob('*.*'))
    for fp in tqdm(files, desc=f'MLLE priors ({split})'):
        out = output_dir / fp.name
        if out.exists():
            continue
        try:
            img = Image.open(fp).convert('RGB')
            generate_mlle_prior(img).save(str(out))
        except Exception as e:
            print(f"  Skipped {fp.name}: {e}")
    print(f"  [{split}] {len(list(output_dir.glob('*.*')))} priors generated")

print("\n MLLE priors ready")

## Step 5 — Clone Repo & Write Model Files

In [ ]:
import os
REPO = '/kaggle/working/uie_repo'
if not os.path.exists(REPO):
    !git clone https://github.com/yanqqq4/Prior-Based-Bi-Encoder-Transformer-for-Underwater-Image-Enhancement.git {REPO}
else:
    print("Repo already cloned")

os.chdir(REPO)
import sys
sys.path.insert(0, REPO)

os.makedirs('models',    exist_ok=True)
os.makedirs('datasets',  exist_ok=True)
os.makedirs('losses',    exist_ok=True)
os.makedirs('checkpoints/enhanced', exist_ok=True)

print("✅ Repo ready at", REPO)

In [ ]:
enhanced_model_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from einops import rearrange

def rgb_to_luminance(x):
    r, g, b = x[:,0:1], x[:,1:2], x[:,2:3]
    return 0.2989*r + 0.5870*g + 0.1140*b

class FFTBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_mag   = nn.Sequential(nn.Conv2d(in_channels, out_channels, 1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
        self.conv_phase = nn.Sequential(nn.Conv2d(in_channels, out_channels, 1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
        self.fusion     = nn.Sequential(nn.Conv2d(out_channels*2, out_channels, 1), nn.ReLU(inplace=True))
    def forward(self, x):
        fft       = torch.fft.rfft2(x, norm="ortho")
        magnitude = F.interpolate(torch.abs(fft),   size=x.shape[2:], mode="bilinear", align_corners=False)
        phase     = F.interpolate(torch.angle(fft), size=x.shape[2:], mode="bilinear", align_corners=False)
        return self.fusion(torch.cat([self.conv_mag(magnitude), self.conv_phase(phase)], dim=1))

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, window_size=8):
        super().__init__()
        self.norm_attn = nn.LayerNorm(dim)
        self.attn      = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.mlp       = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.norm_mlp  = nn.LayerNorm(dim)
        self.ws        = window_size
    def forward(self, x):
        B, C, H, W = x.shape
        ws = self.ws
        if H % ws != 0: x = F.pad(x, (0, 0, 0, ws - H % ws))
        if W % ws != 0: x = F.pad(x, (0, ws - W % ws, 0, 0))
        _, _, Hp, Wp = x.shape
        shortcut  = x
        x_win     = rearrange(x, "b c (h w1) (w w2) -> (b h w) (w1 w2) c", w1=ws, w2=ws)
        tokens    = self.norm_attn(x_win)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        x_out     = rearrange(tokens+attn_out, "(b h w) (w1 w2) c -> b c (h w1) (w w2)", b=B, h=Hp//ws, w=Wp//ws, w1=ws, w2=ws)
        x         = shortcut + x_out
        mlp_in    = rearrange(self.norm_mlp(rearrange(x, "b c h w -> b h w c")), "b h w c -> b c h w")
        mlp_out   = rearrange(self.mlp(mlp_in.flatten(2).transpose(1,2)), "b n c -> b c n").view(B, C, Hp, Wp)
        out       = x + mlp_out
        return out[:, :, :H, :W]

class CNNBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net  = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                                  nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.net(x) + self.skip(x)

class HybridEncoderLevel(nn.Module):
    def __init__(self, in_ch, out_ch, num_heads=4):
        super().__init__()
        self.cnn        = CNNBlock(in_ch, out_ch)
        self.trans      = TransformerBlock(out_ch, num_heads)
        self.fft        = FFTBlock(out_ch, out_ch)
        self.fuse       = nn.Sequential(nn.Conv2d(out_ch*3, out_ch, 1), nn.ReLU(inplace=True))
        self.downsample = nn.MaxPool2d(2)
    def forward(self, x):
        c = self.cnn(x)
        fused = self.fuse(torch.cat([c, self.trans(c), self.fft(c)], dim=1))
        return fused, self.downsample(fused)

class HybridEncoder(nn.Module):
    def __init__(self, in_ch, base_ch=32, num_levels=3, num_heads=4):
        super().__init__()
        self.input_proj = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.levels     = nn.ModuleList()
        ch = base_ch
        for _ in range(num_levels):
            self.levels.append(HybridEncoderLevel(ch, ch*2, num_heads))
            ch *= 2
        self.out_channels = ch
    def forward(self, x):
        x = self.input_proj(x)
        skips = []
        for level in self.levels:
            skip, x = level(x)
            skips.append(skip)
        return skips, x

class GatedBiDirectionalFusion(nn.Module):
    def __init__(self, ch_a, ch_b):
        super().__init__()
        common       = min(ch_a, ch_b)
        self.proj_a  = nn.Conv2d(ch_a, common, 1)
        self.proj_b  = nn.Conv2d(ch_b, common, 1)
        self.gate_a  = nn.Sequential(nn.Conv2d(common*2, common, 1), nn.Sigmoid())
        self.gate_b  = nn.Sequential(nn.Conv2d(common*2, common, 1), nn.Sigmoid())
        self.out_proj= nn.Sequential(nn.Conv2d(common*2, common, 1), nn.ReLU(inplace=True))
        self.out_channels = common
    def forward(self, fa, fb):
        pa, pb = self.proj_a(fa), self.proj_b(fb)
        cat    = torch.cat([pa, pb], dim=1)
        return self.out_proj(torch.cat([self.gate_a(cat)*pa, self.gate_b(cat)*pb], dim=1))

class PhysicsGuidedDecoder(nn.Module):
    def __init__(self, in_ch, skip_channels):
        super().__init__()
        self.up_blocks = nn.ModuleList()
        ch = in_ch
        for skip_ch in reversed(skip_channels):
            out_ch = max(ch//2, 32)
            self.up_blocks.append(nn.Sequential(
                nn.ConvTranspose2d(ch+skip_ch, out_ch, 2, stride=2),
                nn.ReLU(inplace=True),
                CNNBlock(out_ch, out_ch)
            ))
            ch = out_ch
        self.head_J = nn.Sequential(nn.Conv2d(ch, 3, 1), nn.Sigmoid())
        self.head_T = nn.Sequential(nn.Conv2d(ch, 1, 1), nn.Sigmoid())
        self.head_A = nn.Sequential(nn.Conv2d(ch, 3, 1), nn.Sigmoid())
    def forward(self, x, skips):
        for i, block in enumerate(self.up_blocks):
            skip = skips[-(i+1)]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = block(torch.cat([x, skip], dim=1))
        return self.head_J(x), self.head_T(x), self.head_A(x)

class PhysicalReconstructionLayer(nn.Module):
    def forward(self, J, T, A):
        return J*T + A*(1.0 - T)

class EnhancedUIENet(nn.Module):
    def __init__(self, base_ch=32, num_levels=3, num_heads=4):
        super().__init__()
        self.encoder_a     = HybridEncoder(1, base_ch, num_levels, num_heads)
        self.encoder_b     = HybridEncoder(3, base_ch, num_levels, num_heads)
        enc_out_ch         = self.encoder_a.out_channels
        self.fusion        = GatedBiDirectionalFusion(enc_out_ch, enc_out_ch)
        fused_ch           = self.fusion.out_channels
        skip_chs           = [base_ch * (2**(i+1)) * 2 for i in range(num_levels)]
        self.decoder       = PhysicsGuidedDecoder(fused_ch, skip_chs)
        self.reconstruction= PhysicalReconstructionLayer()
    def forward(self, x_rgb, x_prior):
        x_lum                     = rgb_to_luminance(x_rgb)
        skips_a, bottleneck_a     = self.encoder_a(x_lum)
        skips_b, bottleneck_b     = self.encoder_b(x_prior)
        fused                     = self.fusion(bottleneck_a, bottleneck_b)
        merged_skips              = [torch.cat([a, b], dim=1) for a, b in zip(skips_a, skips_b)]
        J, T, A                   = self.decoder(fused, merged_skips)
        H, W = x_rgb.shape[2:]
        if J.shape[2:] != (H, W):
            J = F.interpolate(J, (H,W), mode="bilinear", align_corners=False)
            T = F.interpolate(T, (H,W), mode="bilinear", align_corners=False)
            A = F.interpolate(A, (H,W), mode="bilinear", align_corners=False)
        return self.reconstruction(J, T, A), J, T, A
'''

with open('models/Enhanced_m.py', 'w') as f:
    f.write(enhanced_model_code)

import subprocess
result = subprocess.run(['python', 'models/Enhanced_m.py'], capture_output=True, text=True, cwd=REPO)
print(result.stdout or "(no output — add __main__ block if you want a self-test)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-600:])
else:
    print("✅ models/Enhanced_m.py written successfully")

In [ ]:

dataset_code = """
import os, random
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF

class UIEBDataset(Dataset):
    def __init__(self, root, split='train', img_size=256):
        self.input_dir = os.path.join(root, split, 'INPUT')
        self.gt_dir    = os.path.join(root, split, 'GT')
        self.prior_dir = os.path.join(root, split, 'PRIOR')
        self.img_size  = img_size
        self.split     = split

        def basenames(folder):
            return {os.path.splitext(f)[0]: f
                    for f in os.listdir(folder)
                    if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))}

        inp_map   = basenames(self.input_dir)
        gt_map    = basenames(self.gt_dir)
        prior_map = basenames(self.prior_dir)
        common    = set(inp_map) & set(gt_map) & set(prior_map)
        self.triplets = sorted([(inp_map[b], gt_map[b], prior_map[b]) for b in common])
        print(f'[{split}] {len(self.triplets)} valid triplets')

    def __len__(self): return len(self.triplets)

    def _load(self, path): return Image.open(path).convert('RGB')

    def __getitem__(self, idx):
        inf, gtf, prf = self.triplets[idx]
        inp   = self._load(os.path.join(self.input_dir, inf))
        gt    = self._load(os.path.join(self.gt_dir,    gtf))
        prior = self._load(os.path.join(self.prior_dir, prf))
        rsz   = T.Resize((self.img_size+32, self.img_size+32), interpolation=T.InterpolationMode.BICUBIC)
        inp, gt, prior = rsz(inp), rsz(gt), rsz(prior)
        if self.split == 'train':
            if random.random() > 0.5: inp, gt, prior = TF.hflip(inp), TF.hflip(gt), TF.hflip(prior)
            if random.random() > 0.5: inp, gt, prior = TF.vflip(inp), TF.vflip(gt), TF.vflip(prior)
            i,j,h,w = T.RandomCrop.get_params(inp, (self.img_size, self.img_size))
            inp, gt, prior = TF.crop(inp,i,j,h,w), TF.crop(gt,i,j,h,w), TF.crop(prior,i,j,h,w)
        else:
            crop = T.CenterCrop(self.img_size)
            inp, gt, prior = crop(inp), crop(gt), crop(prior)
        to_t = T.ToTensor()
        return to_t(inp), to_t(prior), to_t(gt)
"""

with open('datasets/uieb_dataset.py', 'w') as f:
    f.write(dataset_code)
print(" datasets/uieb_dataset.py written")

In [ ]:

loss_code = """
import torch, torch.nn as nn, torch.nn.functional as F

class SSIMLoss(nn.Module):
    def __init__(self, ws=11):
        super().__init__()
        self.ws = ws
    def _window(self, size, device):
        coords = torch.arange(size, dtype=torch.float32, device=device) - size//2
        g = torch.exp(-(coords**2)/(2*1.5**2)); g /= g.sum()
        return g.outer(g).unsqueeze(0).unsqueeze(0)
    def forward(self, p, t):
        C1, C2, pad = 0.01**2, 0.03**2, self.ws//2
        w = self._window(self.ws, p.device).expand(p.size(1),-1,-1,-1)
        mu1 = F.conv2d(p,w,padding=pad,groups=p.size(1))
        mu2 = F.conv2d(t,w,padding=pad,groups=p.size(1))
        s1  = F.conv2d(p*p,w,padding=pad,groups=p.size(1)) - mu1**2
        s2  = F.conv2d(t*t,w,padding=pad,groups=p.size(1)) - mu2**2
        s12 = F.conv2d(p*t,w,padding=pad,groups=p.size(1)) - mu1*mu2
        return 1 - ((2*mu1*mu2+C1)*(2*s12+C2)/((mu1**2+mu2**2+C1)*(s1+s2+C2))).mean()

class TotalUIELoss(nn.Module):
    def __init__(self): super().__init__(); self.ssim = SSIMLoss()
    def forward(self, I_rec, J, T, A, gt, I_input):
        l1   = F.l1_loss(I_rec, gt)
        l2   = F.mse_loss(I_rec, gt)
        ssim = self.ssim(I_rec, gt)
        phy  = F.l1_loss(J*T + A*(1-T), I_input)
        j_l1 = F.l1_loss(J, gt)
        total = 0.8*l1 + 0.2*l2 + 0.3*ssim + 0.1*phy + 0.1*j_l1
        return total, {'l1':l1.item(),'l2':l2.item(),'ssim':ssim.item(),'physics':phy.item()}
"""

with open('losses/enhanced_loss.py', 'w') as f:
    f.write(loss_code)
print(" losses/enhanced_loss.py written")

## Step 6 — Quick Sanity Check (Can give OOM error)
Avoid and directly start the training steps

## Step 7 — Training


In [ ]:
import sys, os, torch, torch.nn as nn, numpy as np
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path

for m in list(sys.modules.keys()):
    if any(x in m for x in ['Enhanced_m','uieb_dataset','enhanced_loss']):
        del sys.modules[m]

from models.Enhanced_m     import EnhancedUIENet
from datasets.uieb_dataset import UIEBDataset
from losses.enhanced_loss  import TotalUIELoss

DATA_ROOT   = '/kaggle/working/uieb-dataset'
SAVE_DIR    = 'checkpoints/enhanced'
IMG_SIZE    = 256
BATCH_SIZE  = 4       
NUM_EPOCHS  = 180
LR          = 2e-4
BASE_CH     = 32
NUM_LEVELS  = 3
NUM_HEADS   = 4
RESUME      = None    
# ──────────────────────────────────────

def psnr(pred, target):
    mse = ((pred - target)**2).mean().item()
    return 100.0 if mse == 0 else 10*np.log10(1.0/mse)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

train_ds = UIEBDataset(DATA_ROOT, 'train', IMG_SIZE)
val_ds   = UIEBDataset(DATA_ROOT, 'val',   IMG_SIZE)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=2,          shuffle=False, num_workers=2)

model     = EnhancedUIENet(BASE_CH, NUM_LEVELS, NUM_HEADS).to(device)
criterion = TotalUIELoss().to(device)
optimizer = Adam(model.parameters(), lr=LR, betas=(0.9, 0.999))
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LR*0.01)

print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

start_epoch, best_psnr = 0, 0.0
if RESUME and os.path.exists(RESUME):
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    best_psnr   = ckpt.get('best_psnr', 0.0)
    print(f"Resumed from epoch {start_epoch}")

history = {'train_loss': [], 'val_psnr': []}

for epoch in range(start_epoch, NUM_EPOCHS):
  # for training data
    model.train()
    epoch_loss = 0.0
    for batch_idx, (inp, prior, gt) in enumerate(train_dl):
        inp, prior, gt = inp.to(device), prior.to(device), gt.to(device)
        optimizer.zero_grad()
        I_rec, J, T, A = model(inp, prior)
        loss, ld = criterion(I_rec, J, T, A, gt, inp)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        if (batch_idx+1) % 20 == 0:
            print(f"  E{epoch+1} [{batch_idx+1}/{len(train_dl)}]  loss={loss.item():.4f}  l1={ld['l1']:.3f}  ssim={ld['ssim']:.3f}")

    avg_loss = epoch_loss / len(train_dl)
    history['train_loss'].append(avg_loss)
    scheduler.step()

  # for validation of data
    model.eval()
    val_psnrs = []
    with torch.no_grad():
        for inp, prior, gt in val_dl:
            inp, prior, gt = inp.to(device), prior.to(device), gt.to(device)
            I_rec, *_ = model(inp, prior)
            val_psnrs.append(psnr(I_rec.clamp(0,1), gt))

    avg_psnr = float(np.mean(val_psnrs))
    history['val_psnr'].append(avg_psnr)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]  Loss={avg_loss:.4f}  ValPSNR={avg_psnr:.2f}dB  LR={scheduler.get_last_lr()[0]:.6f}")

    if avg_psnr > best_psnr:
        best_psnr = avg_psnr
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'best_psnr': best_psnr},
                   f'{SAVE_DIR}/best_model.pth')
        print(f"  Saved best model  PSNR={best_psnr:.2f}dB")

    if (epoch+1) % 10 == 0:
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'best_psnr': best_psnr},
                   f'{SAVE_DIR}/epoch_{epoch+1}.pth')
        print(f"  Checkpoint saved: epoch_{epoch+1}.pth")

print(f"\n Training complete! Best PSNR: {best_psnr:.2f}dB")

## Step 8 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss']); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.grid(True)
ax2.plot(history['val_psnr']);   ax2.set_title('Val PSNR (dB)'); ax2.set_xlabel('Epoch'); ax2.grid(True)
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print(f"Best PSNR: {max(history['val_psnr']):.2f} dB")

## Step 9 — Evaluate on Test Set

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity  as compare_ssim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = EnhancedUIENet(BASE_CH, NUM_LEVELS, NUM_HEADS).to(device)
ckpt   = torch.load(f'{SAVE_DIR}/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f"Loaded best model — epoch {ckpt['epoch']+1}, PSNR={ckpt['best_psnr']:.2f}dB")

test_ds = UIEBDataset(DATA_ROOT, 'test', IMG_SIZE)
test_dl = DataLoader(test_ds, batch_size=1, shuffle=False)
psnr_list, ssim_list = [], []

with torch.no_grad():
    for inp, prior, gt in test_dl:
        inp, prior, gt = inp.to(device), prior.to(device), gt.to(device)
        I_rec, *_ = model(inp, prior)
        I_rec = I_rec.clamp(0,1)
        pred_np = I_rec[0].cpu().numpy().transpose(1,2,0)
        gt_np   = gt[0].cpu().numpy().transpose(1,2,0)
        psnr_list.append(compare_psnr(gt_np, pred_np, data_range=1.0))
        ssim_list.append(compare_ssim(gt_np, pred_np, data_range=1.0, channel_axis=2))

print(f"\nTest Results ({len(psnr_list)} images):")
print(f"  PSNR : {np.mean(psnr_list):.2f} ± {np.std(psnr_list):.2f} dB")
print(f"  SSIM : {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")

## Step 10 — Visual Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 6, figsize=(20, 14))
col_titles = ['Input', 'Prior (MLLE)', 'Clean J', 'Trans. T', 'Global A', 'Reconstructed']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=11, fontweight='bold')

sample_indices = np.random.choice(len(test_ds), min(4, len(test_ds)), replace=False)

with torch.no_grad():
    for row, idx in enumerate(sample_indices):
        inp, prior, gt = test_ds[idx]
        I_rec, J, T, A = model(inp.unsqueeze(0).to(device), prior.unsqueeze(0).to(device))
        I_rec = I_rec.clamp(0,1)
        def to_np(t): return t[0].cpu().numpy().transpose(1,2,0)
        imgs = [inp.numpy().transpose(1,2,0), prior.numpy().transpose(1,2,0),
                to_np(J), np.repeat(to_np(T), 3, axis=2), to_np(A), to_np(I_rec)]
        p = compare_psnr(gt.numpy().transpose(1,2,0), to_np(I_rec), data_range=1.0)
        for col, (ax, img) in enumerate(zip(axes[row], imgs)):
            ax.imshow(np.clip(img, 0, 1)); ax.axis('off')
            if col == 5: ax.set_title(f'PSNR={p:.1f}dB', fontsize=9)

plt.suptitle('Enhanced UIE — Physics-Guided Fusion (300 Epochs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/visual_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: /kaggle/working/visual_results.png")